# Simple RAG Pipeline — Demo

**RAG = Retrieval-Augmented Generation.**

Instead of relying only on what an LLM "remembers" from training, we:

1. Take a document (a PDF, in this case a famous ML paper), and split it into small chunks.
2. Turn every chunk into a vector (embedding) that captures its meaning.
3. When the user asks a question, embed the question too, and find the chunks whose vectors are closest (most similar) to it — this is **retrieval**.
4. Stuff those chunks into the LLM's prompt as context, and ask it to answer using only that context — this is **augmented generation**.

```
  PDF ──► chunks ──► embeddings ──► vector store
                                        │
  question ──► embedding ──► search ───┘
                                │
                        top-k relevant chunks
                                │
                    prompt = context + question
                                │
                          small open-source LLM
                                │
                              answer
```

**Stack used here (all free / open-source, runs on CPU):**
- Document: ["Attention Is All You Need"](https://arxiv.org/abs/1706.03762) — the famous Transformer paper (public PDF from arXiv).
- Embeddings: `sentence-transformers/all-MiniLM-L6-v2` (small, fast, great for demos).
- Generator LLM: `google/flan-t5-base` (250M params, instruction-tuned, small enough for CPU).
- Vector search: plain NumPy cosine similarity (no extra vector DB needed for a demo this small).

In [1]:
# Install dependencies (safe to re-run)
%pip install -q transformers sentence-transformers pypdf requests numpy

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
llmcompressor 0.12.0 requires compressed-tensors==0.17.1, but you have compressed-tensors 0.17.0 which is incompatible.
llmcompressor 0.12.0 requires datasets<=5.0.0,>=4.8.4, but you have datasets 5.0.1 which is incompatible.
llmcompressor 0.12.0 requires transformers<=5.10.1,>=5.9.0, but you have transformers 4.57.6 which is incompatible.
vllm 0.24.0 requires transformers>=5.5.3, but you have transformers 4.57.6 which is incompatible.
unbabel-comet 2.2.7 requires numpy<2.0.0,>=1.20.0, but you have numpy 2.3.5 which is incompatible.
unbabel-comet 2.2.7 requires protobuf<5.0.0,>=4.24.4, but you have protobuf 6.33.6 which is incompatible.

[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Step 1 — Get a knowledge source

We download a famous, public-domain-accessible PDF: the original Transformer paper. Swap the URL for any PDF (a textbook chapter, a book, your own notes) and the rest of the pipeline works unchanged.

In [2]:
import requests

PDF_URL = "https://arxiv.org/pdf/1706.03762.pdf"  # "Attention Is All You Need"
PDF_PATH = "attention_is_all_you_need.pdf"

response = requests.get(PDF_URL, timeout=30)
response.raise_for_status()
with open(PDF_PATH, "wb") as f:
    f.write(response.content)

print(f"Downloaded {len(response.content) / 1024:.1f} KB to {PDF_PATH}")

Downloaded 2163.3 KB to attention_is_all_you_need.pdf


In [3]:
from pypdf import PdfReader

reader = PdfReader(PDF_PATH)
pages_text = [page.extract_text() or "" for page in reader.pages]
full_text = "\n".join(pages_text)

print(f"Extracted {len(reader.pages)} pages, {len(full_text)} characters total.")
print("\n--- Preview of first 500 characters ---\n")
print(full_text[:500])

Extracted 15 pages, 39601 characters total.

--- Preview of first 500 characters ---

Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.edu
Łukasz 


## Step 2 — Chunk the text

LLMs and embedding models have limited context, and small, focused chunks retrieve more precisely than giant ones. A common simple strategy: split by words into fixed-size chunks with a little overlap so we don't cut sentences awkwardly across chunk boundaries.

In [4]:
def chunk_text(text, chunk_size=150, overlap=30):
    """Split text into overlapping chunks of `chunk_size` words."""
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        if chunk.strip():
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

chunks = chunk_text(full_text, chunk_size=150, overlap=30)
print(f"Created {len(chunks)} chunks.")
print("\n--- Example chunk #5 ---\n")
print(chunks[5])

Created 51 chunks.

--- Example chunk #5 ---

but a few cases [27], however, such attention mechanisms are used in conjunction with a recurrent network. In this work we propose the Transformer, a model architecture eschewing recurrence and instead relying entirely on an attention mechanism to draw global dependencies between input and output. The Transformer allows for significantly more parallelization and can reach a new state of the art in translation quality after being trained for as little as twelve hours on eight P100 GPUs. 2 Background The goal of reducing sequential computation also forms the foundation of the Extended Neural GPU [16], ByteNet [18] and ConvS2S [9], all of which use convolutional neural networks as basic building block, computing hidden representations in parallel for all input and output positions. In these models, the number of operations required to relate signals from two arbitrary input or output positions grows in the distance between positions, linearly 

## Step 3 — Embed the chunks

We use a small open-source sentence-embedding model to turn each text chunk into a vector. Chunks with similar meaning end up with similar vectors.

In [5]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

chunk_embeddings = embedder.encode(
    chunks, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=True
)

print(f"Embeddings shape: {chunk_embeddings.shape}  (num_chunks x embedding_dim)")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Embeddings shape: (51, 384)  (num_chunks x embedding_dim)


## Step 4 — Retrieval

This is the "R" in RAG. We embed the user's question with the same model, then rank chunks by cosine similarity to find the most relevant ones. Because the embeddings are already normalized, cosine similarity is just a dot product.

In [6]:
import numpy as np

def retrieve(question, top_k=3):
    query_embedding = embedder.encode([question], convert_to_numpy=True, normalize_embeddings=True)[0]
    similarities = chunk_embeddings @ query_embedding  # cosine similarity (dot product of normalized vectors)
    top_indices = np.argsort(similarities)[::-1][:top_k]
    return [(chunks[i], float(similarities[i])) for i in top_indices]

# Quick sanity check
for chunk, score in retrieve("What is self-attention?", top_k=3):
    print(f"[score={score:.3f}] {chunk[:150]}...\n")

[score=0.491] output positions. In these models, the number of operations required to relate signals from two arbitrary input or output positions grows in the dista...

[score=0.434] sinusoidal version because it may allow the model to extrapolate to sequence lengths longer than the ones encountered during training. 4 Why Self-Atte...

[score=0.427] h = 8 parallel attention layers, or heads. For each of these we use dk = dv = dmodel/h = 64. Due to the reduced dimension of each head, the total comp...



## Step 5 — Generation with a small open-source LLM

This is the "G" in RAG. `google/flan-t5-base` is an instruction-tuned model small enough to run comfortably on CPU, which makes it good for classroom demos (no GPU required, no API key, no cost).

In [7]:
from transformers import pipeline

generator = pipeline("text2text-generation", model="google/flan-t5-base")

# Quick sanity check — generation with NO retrieved context, just the raw model
print(generator("What is the capital of France?", max_new_tokens=50)[0]["generated_text"])

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


london


## Step 6 — Put it all together: the RAG function

Retrieve the most relevant chunks, drop them into a prompt as context, and ask the LLM to answer strictly from that context. This is what makes the answer grounded in the document instead of the model's generic training knowledge.

In [8]:
def rag_answer(question, top_k=3, max_new_tokens=100, verbose=True):
    retrieved = retrieve(question, top_k=top_k)
    context = "\n\n".join(chunk for chunk, _ in retrieved)

    prompt = (
        "Answer the question using only the context below. "
        "If the context doesn't contain the answer, say so.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}\n"
        "Answer:"
    )

    result = generator(prompt, max_new_tokens=max_new_tokens)[0]["generated_text"]

    if verbose:
        print("Retrieved chunks:")
        for chunk, score in retrieved:
            print(f"  [score={score:.3f}] {chunk[:120]}...")
        print(f"\nQuestion: {question}")
        print(f"Answer: {result}")

    return result

## Demo — ask the paper some questions

In [9]:
demo_questions = [
    "What is the Transformer architecture based on?",
    "What are the two main components of the encoder-decoder structure?",
    "Why do the authors avoid recurrence and convolutions?",
]

for q in demo_questions:
    rag_answer(q)
    print("\n" + "=" * 80 + "\n")

Token indices sequence length is longer than the specified maximum sequence length for this model (770 > 512). Running this sequence through the model will result in indexing errors


Retrieved chunks:
  [score=0.501] an input sequence of symbol representations (x1, ..., xn) to a sequence of continuous representations z = (z1, ..., zn)....
  [score=0.443] h = 8 parallel attention layers, or heads. For each of these we use dk = dv = dmodel/h = 64. Due to the reduced dimensio...
  [score=0.392] Transformer architecture. Unlisted values are identical to those of the base model. All metrics are on the English-to-Ge...

Question: What is the Transformer architecture based on?
Answer: The Transformer follows this overall architecture using stacked self-attention and point-wise, fully connected layers for both the encoder and decoder, shown in the left and right halves of Figure 1, respectively. 3.1 Encoder and Decoder Stacks Encoder: The encoder is composed of a stack of N = 6 identical layers. Each layer has two sub-layers. The first is a multi-head self-attention mechanism, and the second is


Retrieved chunks:
  [score=0.546] two sub-layers. The first is a multi-head s

## Try it yourself

Change the question below and re-run — notice how the answer stays grounded in whatever chunks get retrieved.

In [ ]:
your_question = "What dataset was used to train the Transformer model?"
rag_answer(your_question)

## Discussion / exercises for students

1. **Swap the document.** Replace `PDF_URL` with any other PDF (a book chapter, lecture notes) — the pipeline doesn't change.
2. **Change `chunk_size` and `overlap`.** Smaller chunks retrieve more precisely but lose surrounding context; larger chunks do the opposite. Try both and compare answers.
3. **Change `top_k` in `retrieve`.** What happens to answer quality with `top_k=1` vs `top_k=5`?
4. **Ask a question the document can't answer** (e.g. "What's the weather today?") — a good RAG system should say it doesn't know rather than hallucinate.
5. **Compare with vs without retrieval.** Call `generator(question, ...)` directly (no context) versus `rag_answer(question)` — this shows exactly what retrieval adds.
6. **Scale up (optional).** Swap `all-MiniLM-L6-v2` → a larger embedding model, or `flan-t5-base` → `flan-t5-large`, and swap the NumPy search for a real vector DB (e.g. FAISS, Chroma) once the corpus grows beyond a single paper.